# E5 · Khả năng mở rộng · Chủ: Người 1 · Phục vụ Chương 4 (mục 4.3)
Thời gian huấn luyện theo kích thước dữ liệu → **T4.2 / F4.2**; tốc độ đọc CSV và Parquet → **T4.3**.

**Bản v2 (05/11)** sửa hai điểm yếu của lần chạy 1 (kết quả cũ được giữ với hậu tố `_v1`):
- T4.2: mỗi cỡ đo **3 lần** (lần 1 chỉ đo 1 lần, mà cùng 58.987 dòng E4 mất 442s còn E5 mất 363s ⇒ dao động ~20%);
  lấy mẫu theo `hash(id)` để tập con giống hệt nhau ở mọi phiên; khớp đường thẳng để tách chi phí cố định và chi phí theo dòng.
- T4.3: lần 1 so CSV 467.985 dòng với Parquet 58.987 dòng (**khác dữ liệu**, không so được). Nay chuyển **chính file CSV đó** sang Parquet,
  cùng ổ đĩa local, cùng số dòng; đo 3 lần lấy trung vị; ghi cả dung lượng.

Chạy: **Runtime → Run all**. T4.2 mất ~50 phút (4 cỡ × 3 lần), T4.3 ~10 phút.

In [ ]:
# === BOOTSTRAP: giữ nguyên ở đầu mọi notebook ===
OWNER = "Son&Tai"   # tên bạn
REPO  = "https://github.com/thanhsonnguyen-gif/ai-text-detection.git"

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    url = REPO
    try:   # repo private: lưu Personal Access Token vào Colab Secrets với tên GH_TOKEN
        from google.colab import userdata
        url = REPO.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
else:   # --ff-only + check=True: pull hỏng thì DỪNG, không âm thầm chạy tiếp với code cũ
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q", "--ff-only"], check=True)
print("Code đang dùng:", subprocess.run(["git", "-C", "/content/repo", "log", "-1", "--format=%h %s (%cd)", "--date=short"],
                                        capture_output=True, text=True).stdout.strip())
for k in [k for k in sys.modules if k == "aitext" or k.startswith("aitext.")]:
    del sys.modules[k]   # chạy lại BOOTSTRAP ⇒ nạp lại aitext mới nhất, không dùng bản cũ còn trong bộ nhớ
if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/repo/requirements.txt"])

from aitext import config, spark_utils
spark_utils.ensure_pyspark()
spark = spark_utils.get_spark()
config.ensure_dirs()
import time, pandas as pd, matplotlib.pyplot as plt
from pyspark.sql import functions as F
print("Drive:", config.DRIVE_ROOT)

In [ ]:
# 0. Lưu vết lần chạy 1 trước khi ghi đè (chỉ đổi tên một lần)
import os, shutil, subprocess, numpy as np
from aitext import data, models, evaluate
for f in ["T4.2_scaling.csv", "T4.3_csv_vs_parquet.csv", "T4.3_csv_vs_parquet_notes.md", "figures/F4.2_scaling.png"]:
    src = f"{config.RESULTS_DIR}/{f}"; root, ext = os.path.splitext(src); dst = f"{root}_v1{ext}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.move(src, dst); print("đã đổi tên:", dst)

In [ ]:
# 1. T4.2: thời gian huấn luyện theo số dòng, 3 lần mỗi cỡ
train, _, _ = data.load_splits(spark)
train = data.with_class_weight(train)
REPEATS = 3; rows = []
for frac in [0.1, 0.25, 0.5, 1.0]:
    # lấy mẫu theo hash(id): cùng một tập con ở mọi phiên Spark (train.sample không tái lập giữa các phiên)
    part = train.filter(F.abs(F.hash("id")) % 1000 < int(frac * 1000)).cache(); n = part.count()
    for rep in range(1, REPEATS + 1):
        t0 = time.time(); models.fit(part, defended=True); s = time.time() - t0
        rows.append({"số dòng": n, "lần": rep, "giây": s}); print(f"{n:>6} dòng, lần {rep}: {s:.0f}s")
    part.unpersist()
t42raw = pd.DataFrame(rows); evaluate.save_table(t42raw, "T4.2_scaling_raw")
t42 = t42raw.groupby("số dòng")["giây"].agg(["mean", "std", "min"]).reset_index()

# Khớp đường thẳng trên các cỡ >= 14.000 dòng: giây ≈ cố định + hệ số × số dòng
big = t42[t42["số dòng"] >= 14_000]
slope, icpt = np.polyfit(big["số dòng"], big["mean"], 1)
FULL = 467_985   # toàn bộ train_none (người + AI, trước khi lấy mẫu)
notes = (f"{REPEATS} lần đo mỗi cỡ; Colab CPU {os.cpu_count()} lõi, Spark local[*]; LR maxIter=30, 2×2^18 đặc trưng. "
         f"Khớp tuyến tính (>= 14k dòng): ~{icpt:.0f}s cố định + {slope * 1000:.2f}s mỗi 1.000 dòng. "
         f"Ngoại suy cho toàn bộ train_none ({FULL:,} dòng): ~{(icpt + slope * FULL) / 60:.0f} phút (chỉ ước lượng, ngoài vùng đã đo).")
display(t42.round(1)); print(notes)
evaluate.save_table(t42, "T4.2_scaling", notes=notes)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.errorbar(t42["số dòng"], t42["mean"], yerr=t42["std"], marker="o", capsize=4, label="đo (TB ± std, 3 lần)")
xs = np.linspace(0, t42["số dòng"].max(), 50)
ax.plot(xs, icpt + slope * xs, "--", alpha=0.6, label=f"{icpt:.0f}s + {slope * 1000:.2f}s/1k dòng")
ax.set_xlabel("số dòng huấn luyện"); ax.set_ylabel("giây"); ax.set_ylim(bottom=0); ax.legend()
ax.set_title("F4.2 Thời gian huấn luyện theo số dòng"); evaluate.save_figure(fig, "F4.2_scaling"); plt.show()

## T4.3: CSV và Parquet trên CÙNG dữ liệu (cần tải lại `train_none.csv`, khoảng 5 phút)

In [ ]:
# 2. T4.3: cùng 467.985 dòng, cùng ổ local; 3 lần mỗi tác vụ, lấy trung vị
csv = data.download("train_none")
pq = f"{config.LOCAL_TMP}/bench/train_none_parquet"
if not os.path.exists(pq):
    # CSV multiLine không chia nhỏ được ⇒ đọc bằng 1 luồng; chia 8 phần khi ghi Parquet để đọc song song
    data.read_raid_csv(spark, csv).repartition(8).write.mode("overwrite").parquet(pq)
du = lambda p: int(subprocess.run(["du", "-sb", p], capture_output=True, text=True).stdout.split()[0])
load = {"CSV": lambda: data.read_raid_csv(spark, csv), "Parquet": lambda: spark.read.parquet(pq)}
n_rows = {k: f().count() for k, f in load.items()}
assert n_rows["CSV"] == n_rows["Parquet"], n_rows      # bảo đảm so sánh trên cùng dữ liệu
tasks = {"count": lambda d: d.count(),
         "groupBy model (1 cột nhỏ)": lambda d: d.groupBy("model").count().collect(),
         "độ dài TB theo nhãn (cột văn bản)": lambda d: d.groupBy("label").agg(F.avg(F.length("generation"))).collect()}
rows = []
for fmt, f in load.items():
    for task, fn in tasks.items():
        ts = []
        for _ in range(3):
            t0 = time.time(); fn(f()); ts.append(time.time() - t0)
        rows.append({"tác vụ": task, "định dạng": fmt, "giây": float(np.median(ts))})
t43 = pd.DataFrame(rows).pivot(index="tác vụ", columns="định dạng", values="giây").reset_index()
t43["Parquet nhanh hơn (×)"] = t43["CSV"] / t43["Parquet"]
size = {"CSV": du(csv), "Parquet": du(pq)}
notes = (f"Cùng {n_rows['CSV']:,} dòng train_none, cùng ổ local Colab, trung vị 3 lần. "
         f"Dung lượng: CSV {size['CSV'] / 1e6:.0f} MB, Parquet {size['Parquet'] / 1e6:.0f} MB "
         f"(nhỏ hơn {size['CSV'] / size['Parquet']:.1f}×). count trên Parquet đọc số dòng trong metadata, không quét dữ liệu.")
display(t43.round(3)); print(notes)
evaluate.save_table(t43, "T4.3_csv_vs_parquet", notes=notes)

## Ghi chú cho Chương 4
- ...